## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
from xgboost import XGBClassifier
import joblib
import warnings
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

sns.set_style('darkgrid')
warnings.filterwarnings('ignore')

## 2. Dataset Loading

In [ ]:
df = pd.read_csv('stroke_data.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Exploratory Data Analysis

In [ ]:
df.isnull().sum()

In [ ]:
fig = px.pie(df, names='stroke', title='Stroke Distribution')
fig.show()

In [ ]:
fig = px.histogram(df, x='age', color='stroke', barmode='overlay', title='Age Distribution by Stroke')
fig.show()

In [ ]:
fig = px.box(df, x='stroke', y='avg_glucose_level', title='Avg Glucose Level by Stroke')
fig.show()

In [ ]:
fig = px.box(df, x='stroke', y='bmi', title='BMI by Stroke')
fig.show()

In [ ]:
gender_stroke = pd.crosstab(df['gender'], df['stroke'])
fig = px.imshow(gender_stroke, text_auto=True, title='Gender vs Stroke')
fig.show()

In [ ]:
fig = px.histogram(df, x='smoking_status', color='stroke', barmode='group', title='Smoking Status vs Stroke')
fig.show()

In [ ]:
fig = px.histogram(df, x='work_type', color='stroke', barmode='group', title='Work Type vs Stroke')
fig.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## 4. Data Preprocessing

In [ ]:
df_clean = df.drop('id', axis=1)

df_clean['bmi'] = df_clean['bmi'].fillna(df_clean['bmi'].median())

smoking_mode = df_clean['smoking_status'].mode()[0]
df_clean['smoking_status'] = df_clean['smoking_status'].replace('Unknown', smoking_mode)

encoders = {}
for col in ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']:
    encoders[col] = LabelEncoder()
    df_clean[col] = encoders[col].fit_transform(df_clean[col])

df_clean.head()

## 5. Handling Imbalanced Data

In [ ]:
X = df_clean.drop('stroke', axis=1)
y = df_clean['stroke']

print('Class distribution:')
print(y.value_counts())
print()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train distribution:\n{y_train.value_counts()}')
print(f'y_test distribution:\n{y_test.value_counts()}')

In [ ]:
numeric_cols = ['age', 'avg_glucose_level', 'bmi']
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print('Scaling applied to:', numeric_cols)

In [ ]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f'Before SMOTE - X_train shape: {X_train_scaled.shape}')
print(f'After SMOTE - X_train shape: {X_train_resampled.shape}')
print(f'Before SMOTE - y_train distribution:\n{y_train.value_counts()}')
print(f'After SMOTE - y_train distribution:\n{y_train_resampled.value_counts()}')

## 6. Model Training & Evaluation

In [ ]:
neg = len(y_train[y_train == 0])
pos = len(y_train[y_train == 1])
scale_pos_weight = neg / pos

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=42),
    'XGBoost': XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42)
}

results = []
best_model = None
best_score = 0
best_name = ''

for name, model in models.items():
    print(f'\n{"="*60}')
    print(f'Training {name}...')
    print(f'{"="*60}')
    
    try:
        if name == 'XGBoost':
            model.fit(X_train_resampled, y_train_resampled)
        else:
            model.fit(X_train_resampled, y_train_resampled)
        
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc = roc_auc_score(y_test, y_proba)
        
        results.append({
            'Model': name,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1-Score': f1,
            'ROC-AUC': roc
        })
        
        print(f'Accuracy: {acc:.4f}')
        print(f'Precision: {prec:.4f}')
        print(f'Recall: {rec:.4f}')
        print(f'F1-Score: {f1:.4f}')
        print(f'ROC-AUC: {roc:.4f}')
        print(f'\nClassification Report:')
        print(classification_report(y_test, y_pred))
        
        if roc > best_score:
            best_score = roc
            best_model = model
            best_name = name
            
    except Exception as e:
        print(f'Error training {name}: {e}')

print(f'\n{"="*60}')
print(f'Best Model: {best_name} with ROC-AUC = {best_score:.4f}')
print(f'{"="*60}')

In [ ]:
results_df = pd.DataFrame(results).round(4)
results_df

## 7. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Stroke', 'Stroke'],
                yticklabels=['No Stroke', 'Stroke'])
    ax.set_title(f'{name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 8. ROC Curve Comparison

In [ ]:
plt.figure(figsize=(10, 8))

for name, model in models.items():
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

## 9. Feature Importance

In [ ]:
rf_model = models['Random Forest']
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(feature_importance['Feature'], feature_importance['Importance'])
plt.xlabel('Importance')
plt.title('Feature Importance (Random Forest)')
plt.tight_layout()
plt.show()

## 10. Saving the Best Model

In [ ]:
joblib.dump(best_model, 'brain_stroke_model.pkl')
joblib.dump(scaler, 'brain_stroke_scaler.pkl')
joblib.dump(encoders, 'brain_stroke_encoders.pkl')

print(f'Best model ({best_name}) saved as brain_stroke_model.pkl')
print('Scaler saved as brain_stroke_scaler.pkl')
print('Encoders saved as brain_stroke_encoders.pkl')

## 11. Model Loading & Prediction Demo

In [ ]:
loaded_model = joblib.load('brain_stroke_model.pkl')
loaded_scaler = joblib.load('brain_stroke_scaler.pkl')
loaded_encoders = joblib.load('brain_stroke_encoders.pkl')

sample = pd.DataFrame([{
    'gender': 'Male',
    'age': 45,
    'hypertension': 0,
    'heart_disease': 0,
    'ever_married': 'Yes',
    'work_type': 'Private',
    'Residence_type': 'Urban',
    'avg_glucose_level': 120.0,
    'bmi': 28.5,
    'smoking_status': 'never smoked'
}])

print('Sample Input:')
print(sample.to_string(index=False))
print()

for col in ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']:
    sample[col] = loaded_encoders[col].transform(sample[col])

numeric_cols = ['age', 'avg_glucose_level', 'bmi']
sample[numeric_cols] = loaded_scaler.transform(sample[numeric_cols])

pred = loaded_model.predict(sample)[0]
proba = loaded_model.predict_proba(sample)[0, 1]

print('Prediction Result:')
print(f'{"Stroke" if pred == 1 else "No Stroke"} (Probability: {proba:.4f})')

## 12. Conclusion

This notebook developed and evaluated machine learning models for stroke prediction using demographic, medical, and lifestyle features. Key findings:

- The dataset was highly imbalanced, with far fewer stroke cases than non-stroke cases. SMOTE oversampling was used to address this.
- Three models were compared: Logistic Regression, Random Forest, and XGBoost.
- Performance was evaluated using accuracy, precision, recall, F1-score, and ROC-AUC.
- Feature importance analysis revealed which factors most influence stroke risk.
- The best performing model was saved for future predictions.

This pipeline can be extended with hyperparameter tuning, additional feature engineering, and deployment as a web service or clinical decision support tool.